# Mini Project 1: Word Guessing Game

**PGR211 Advanced Programming for Data Science** — mandatory coursework requirement

## The game

- A random word is chosen at random from a list.
- The player enters their name, then guesses one letter at a time.
- Correct letters are revealed in their correct positions.
- A wrong letter costs one of the 12 turns; the turn count is configurable.
- The round ends when the word is complete or the turns run out.

## How the design covers the Learning Outcomes

| Concept | Where it appears |
| --- | --- |
| Class and object | `WordGuessingGame`, `ConsoleGameRunner` and the word sources are classes; each round is an object |
| Method | `guess`, `masked_word`, `reveal_word`, `run` |
| Inheritance | `FixedWordSource`, `RandomWordSource` and `DifficultyWordSource` derive from `WordSource`; `DifficultyWordSource` extends `RandomWordSource` via `super()` |
| Polymorphism | `WordGuessingGame` accepts any `WordSource` and calls `next_word()` without knowing the concrete type |
| Encapsulation | The secret word, guessed letters and turn counter are private; state is read through properties and changed only by `guess` |

## 1. Imports and configuration

In [ ]:
import random
from __future__ import annotations
from abc import ABC, abstractmethod
from collections.abc import Callable, Iterable
from enum import Enum, auto

DEFAULT_WORDS: tuple[str, ...] = (
    "kristiania",
    "computer",
    "science",
    "programming",
    "python",
    "inheritance",
    "polymorphism",
    "encapsulation",
)

DEFAULT_MAX_WRONG_GUESSES = 12

## 2. Word sources: inheritance and polymorphism

`WordSource` is an abstract base class. It cannot be instantiated and it defines the single operation the game needs: `next_word()`. Shared validation lives in the base class so no subclass has to repeat it.

Three subclasses implement the abstraction differently:

- `FixedWordSource` returns words in a known order, which makes tests deterministic.
- `RandomWordSource` picks at random, and accepts an injected random generator so the randomness can be seeded.
- `DifficultyWordSource` inherits from `RandomWordSource` and narrows the candidates by word length, delegating the rest of the work to `super().__init__`.

This is polymorphism in the useful sense: the game works with all three and knows about none of them.

In [ ]:
class WordSource(ABC):

    @abstractmethod
    def next_word(self) -> str:

    @staticmethod
    def normalise(word: str) -> str:

        cleaned = word.strip().lower()
        if not cleaned:
            raise ValueError("Words must not be empty.")
        if not cleaned.isalpha():
            raise ValueError(f"Words must contain letters only, got {word!r}.")
        return cleaned

    @classmethod
    def _normalise_all(cls, words: Iterable[str]) -> list[str]:
        normalised = [cls.normalise(word) for word in words]
        if not normalised:
            raise ValueError("At least one word is required.")
        return normalised

class FixedWordSource(WordSource):

    def __init__(self, words: Iterable[str]) -> None:
        self._words = self._normalise_all(words)
        self._index = 0

    def next_word(self) -> str:
        word = self._words[self._index % len(self._words)]
        self._index += 1
        return word

class RandomWordSource(WordSource):

    def __init__(self, words: Iterable[str], rng: random.Random | None = None) -> None:
        self._words = self._normalise_all(words)
        self._rng = rng if rng is not None else random.Random()

    @property
    def words(self) -> tuple[str, ...]:
        return tuple(self._words)

    def next_word(self) -> str:
        return self._rng.choice(self._words)

class DifficultyWordSource(RandomWordSource):

    def __init__(
        self,
        words: Iterable[str],
        min_length: int = 1,
        max_length: int | None = None,
        rng: random.Random | None = None,
    ) -> None:
        if min_length < 1:
            raise ValueError("min_length must be at least 1.")
        if max_length is not None and max_length < min_length:
            raise ValueError("max_length must not be smaller than min_length.")

        upper_bound = max_length if max_length is not None else float("inf")
        candidates = [
            word
            for word in self._normalise_all(words)
            if min_length <= len(word) <= upper_bound
        ]
        if not candidates:
            raise ValueError(
                f"No words between {min_length} and {max_length} letters long."
            )
        super().__init__(candidates, rng=rng)

## 3. Value types

`GuessOutcome` and `GameState` replace the loose strings and flags a procedural version would use. An enum makes the set of possibilities explicit, and `GuessOutcome.costs_turn` keeps the scoring rule attached to the outcome rather than scattered through the game loop.

In [ ]:
class GuessOutcome(Enum):

    CORRECT = auto()
    INCORRECT = auto()
    ALREADY_GUESSED = auto()
    INVALID = auto()

    @property
    def costs_turn(self) -> bool:
        return self is GuessOutcome.INCORRECT

class GameState(Enum):

    IN_PROGRESS = auto()
    WON = auto()
    LOST = auto()

## 4. The game: encapsulation

`WordGuessingGame` owns every piece of mutable state. The attributes are private, reads go through read-only properties, and `guess()` is the only way to change anything, so the object cannot be pushed into an inconsistent state from outside.

Two rules are enforced rather than merely documented:

- `reveal_word()` raises if the round is still running, so the interface layer cannot leak the answer by accident.
- `guess()` raises once the round is over, instead of quietly accepting guesses that can no longer matter.

Invalid and repeated guesses are reported back to the caller but cost the player nothing.

In [ ]:
class WordGuessingGame:

    def __init__(
        self,
        word_source: WordSource,
        max_wrong_guesses: int = DEFAULT_MAX_WRONG_GUESSES,
    ) -> None:
        if max_wrong_guesses < 1:
            raise ValueError("max_wrong_guesses must be at least 1.")
        self._secret_word = word_source.next_word()
        self._max_wrong_guesses = max_wrong_guesses
        self._wrong_guesses = 0
        self._guessed_letters: set[str] = set()

    @property
    def word_length(self) -> int:
        return len(self._secret_word)

    @property
    def remaining_turns(self) -> int:
        return self._max_wrong_guesses - self._wrong_guesses

    @property
    def guessed_letters(self) -> frozenset[str]:
        return frozenset(self._guessed_letters)

    @property
    def wrong_letters(self) -> tuple[str, ...]:
        return tuple(
            sorted(
                letter
                for letter in self._guessed_letters
                if letter not in self._secret_word
            )
        )

    @property
    def state(self) -> GameState:
        if self._is_word_complete():
            return GameState.WON
        if self.remaining_turns <= 0:
            return GameState.LOST
        return GameState.IN_PROGRESS

    @property
    def is_over(self) -> bool:
        return self.state is not GameState.IN_PROGRESS

    def masked_word(self, placeholder: str = "_", separator: str = " ") -> str:
        return separator.join(
            letter if letter in self._guessed_letters else placeholder
            for letter in self._secret_word
        )

    def guess(self, letter: str) -> GuessOutcome:
        if self.is_over:
            raise RuntimeError("The round is already over; start a new game.")

        candidate = letter.strip().lower()
        if len(candidate) != 1 or not candidate.isalpha():
            return GuessOutcome.INVALID
        if candidate in self._guessed_letters:
            return GuessOutcome.ALREADY_GUESSED

        self._guessed_letters.add(candidate)
        if candidate in self._secret_word:
            return GuessOutcome.CORRECT

        self._wrong_guesses += 1
        return GuessOutcome.INCORRECT

    def reveal_word(self) -> str:
        if not self.is_over:
            raise RuntimeError("The word can only be revealed once the round is over.")
        return self._secret_word

    def _is_word_complete(self) -> bool:
        return all(letter in self._guessed_letters for letter in self._secret_word)

## 5. Console interface

`ConsoleGameRunner` is the only part of the program that talks to the user. Input and output are injected as functions, defaulting to `input` and `print`, which is what allows the tests below to play whole rounds with scripted keystrokes and assert on what was printed.

In [ ]:
class ConsoleGameRunner:

    _MESSAGES: dict[GuessOutcome, str] = {
        GuessOutcome.CORRECT: "Correct!",
        GuessOutcome.INCORRECT: "Wrong!",
        GuessOutcome.ALREADY_GUESSED: "You already guessed that letter.",
        GuessOutcome.INVALID: "Please enter exactly one letter.",
    }

    def __init__(
        self,
        game: WordGuessingGame,
        player_name: str = "Player",
        input_fn: Callable[[str], str] = input,
        output_fn: Callable[[str], None] = print,
    ) -> None:
        self._game = game
        self._player_name = player_name.strip() or "Player"
        self._input_fn = input_fn
        self._output_fn = output_fn

    def run(self) -> GameState:
        self._announce_start()
        while not self._game.is_over:
            self._show_status()
            guess = self._input_fn("Guess a letter: ")
            outcome = self._game.guess(guess)
            self._output_fn(self._MESSAGES[outcome])
        self._announce_result()
        return self._game.state

    def _announce_start(self) -> None:
        self._output_fn(f"\nGood luck, {self._player_name}!")
        self._output_fn(
            f"The word has {self._game.word_length} letters. "
            "Guess it one letter at a time."
        )

    def _show_status(self) -> None:
        self._output_fn(f"\nWord: {self._game.masked_word()}")
        self._output_fn(f"Turns remaining: {self._game.remaining_turns}")
        wrong_letters = self._game.wrong_letters
        if wrong_letters:
            self._output_fn(f"Wrong letters: {', '.join(wrong_letters)}")

    def _announce_result(self) -> None:
        word = self._game.reveal_word()
        if self._game.state is GameState.WON:
            self._output_fn(f"\nYou win, {self._player_name}! The word was '{word}'.")
        else:
            self._output_fn(f"\nYou lose, {self._player_name}. The word was '{word}'.")

## 6. Entry point

`main()` wires the pieces together. The interactive call is behind a flag so that re-running this notebook top to bottom never blocks on input.

In [ ]:
def main() -> None:
    try:
        name = input("Enter your name: ")
    except (EOFError, KeyboardInterrupt):
        print("\nNo input received; exiting.")
        return

    game = WordGuessingGame(RandomWordSource(DEFAULT_WORDS))
    runner = ConsoleGameRunner(game, player_name=name)

    try:
        runner.run()
    except (EOFError, KeyboardInterrupt):
        print("\nGame interrupted. Goodbye.")

# Set to True and re-run this cell to play a full interactive round.
PLAY_INTERACTIVELY = False

if PLAY_INTERACTIVELY:
    main()